# TA-002 — Anonimizar y Preprocesar Hemogramas
**Proyecto:** Sistema Multimodal de IA para Apoyo Diagnóstico Clínico Veterinario — Vargas Vet  
**Curso:** Taller Integrador 1 — UPAO  
**Integrantes:** Baylón Toledo, Diogho Matteo (PM) · Saavedra Arroyo, Sebastián Alonso (Scrum Master)  

**Objetivo:**  
Anonimizar los hemogramas recibidos de la clínica colaboradora eliminando datos personales del propietario/paciente y renombrando los archivos con IDs anónimos secuenciales, para su uso seguro en las pruebas del módulo de extracción.

**Criterios de aceptación:**
- Datos personales eliminados de cada archivo (metadata PDF)
- Archivos renombrados con ID anónimo secuencial (`HG-XXXX` para PDFs, `VCH-XXXX` para todos los vouchers e imágenes)
- Mapping privado generado para trazabilidad interna (NO subir al repo)

## 1 · Verificación de entorno

In [1]:
import os
import sys
import shutil
import csv
from pathlib import Path

try:
    from pypdf import PdfReader, PdfWriter
    print('pypdf OK:', __import__('pypdf').__version__)
except ImportError:
    print('Instalando pypdf...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pypdf', '-q'])
    from pypdf import PdfReader, PdfWriter
    print('pypdf instalado')

pypdf OK: 6.13.1


## 2 · Rutas y configuración

In [2]:
BASE      = Path(r'E:\Taller Integrador I\ModelosIA\Hemogramas')
PDFS_SRC  = BASE / 'PDFS'
VOUC_SRC  = BASE / 'VOUCHERS'
PDFS_DST  = BASE / 'PDFS_anon'
VOUC_DST  = BASE / 'VOUCHERS_anon'
MAPPING   = BASE / 'hemogramas_mapping_PRIVADO.csv'

for folder in [PDFS_DST, VOUC_DST]:
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir()

print(f'Origen PDFs:      {PDFS_SRC}')
print(f'Origen vouchers:  {VOUC_SRC}')
print(f'Destino PDFs:     {PDFS_DST}')
print(f'Destino vouchers: {VOUC_DST}')
print(f'Mapping privado:  {MAPPING}')

Origen PDFs:      E:\Taller Integrador I\ModelosIA\Hemogramas\PDFS
Origen vouchers:  E:\Taller Integrador I\ModelosIA\Hemogramas\VOUCHERS
Destino PDFs:     E:\Taller Integrador I\ModelosIA\Hemogramas\PDFS_anon
Destino vouchers: E:\Taller Integrador I\ModelosIA\Hemogramas\VOUCHERS_anon
Mapping privado:  E:\Taller Integrador I\ModelosIA\Hemogramas\hemogramas_mapping_PRIVADO.csv


## 3 · Inventario inicial

In [3]:
pdfs          = sorted([f for f in os.listdir(PDFS_SRC) if f.lower().endswith('.pdf')])
jpg_misplaced = [f for f in os.listdir(PDFS_SRC) if not f.lower().endswith('.pdf')]

vouc_all      = os.listdir(VOUC_SRC)
vouc_whatsapp = sorted([f for f in vouc_all if f.startswith('WhatsApp')])
vouc_uuid     = sorted([f for f in vouc_all if not f.startswith('WhatsApp')])

print('=== INVENTARIO INICIAL ===')
print(f'PDFs con nombre real:     {len(pdfs)}')
print(f'Archivos extra en PDFS/: {len(jpg_misplaced)} → {jpg_misplaced}')
print(f'Vouchers UUID:            {len(vouc_uuid)}')
print(f'Imágenes WhatsApp:        {len(vouc_whatsapp)}')
print()
print('Primeros 5 PDFs (datos personales):')
for f in pdfs[:5]:
    print(f'  {f}')
print('  ...')
print()
print('Imágenes WhatsApp:')
for f in vouc_whatsapp:
    print(f'  {f}')

=== INVENTARIO INICIAL ===
PDFs con nombre real:     1084
Archivos extra en PDFS/: 0 → []
Vouchers UUID:            31
Imágenes WhatsApp:        5

Primeros 5 PDFs (datos personales):
  AARON-BLAS-2024-06-27-1930.pdf
  ABBY-HORNA-2025-07-09-1528.pdf
  ABBY-HORNA-2025-07-17-2150.pdf
  ABRIL-MUÑOZ-2024-04-23-1835.pdf
  ADA-CABOS-2026-01-04-1218.pdf
  ...

Imágenes WhatsApp:
  WhatsApp Image 2026-06-06 at 23.05.28.jpeg
  WhatsApp Image 2026-06-06 at 23.05.42.jpeg
  WhatsApp Image 2026-06-06 at 23.05.56.jpeg
  WhatsApp Image 2026-06-06 at 23.06.26.jpeg
  WhatsApp Image 2026-06-06 at 23.06.35.jpeg


## 4 · Anonimización de PDFs
> Se copia cada PDF a `PDFS_anon/` con nombre `HG-XXXX.pdf`, eliminando la metadata del archivo (autor, título, creador, fecha) mediante pypdf.

In [4]:
mapping_rows = []
errores      = []

print(f'Procesando {len(pdfs)} PDFs...')

for i, fname in enumerate(pdfs, start=1):
    nuevo = f'HG-{i:04d}.pdf'
    src   = PDFS_SRC / fname
    dst   = PDFS_DST / nuevo

    try:
        reader = PdfReader(str(src))
        writer = PdfWriter()
        for page in reader.pages:
            writer.add_page(page)
        writer.add_metadata({})
        with open(dst, 'wb') as f:
            writer.write(f)
    except Exception as e:
        shutil.copy2(src, dst)
        errores.append({'archivo': fname, 'error': str(e)})

    mapping_rows.append({
        'tipo': 'PDF',
        'nombre_original': fname,
        'nombre_anonimo': nuevo
    })

    if i % 200 == 0:
        print(f'  {i}/{len(pdfs)} procesados...')

print(f'PDFs completados: {len(pdfs)}')
if errores:
    print(f'Errores de metadata (copiados sin limpiar): {len(errores)}')
    for e in errores[:5]:
        print(f'  {e["archivo"]}: {e["error"]}')

Procesando 1084 PDFs...
  200/1084 procesados...
  400/1084 procesados...
  600/1084 procesados...
  800/1084 procesados...
  1000/1084 procesados...
PDFs completados: 1084


## 5 · Anonimizar vouchers (todos con VCH-XXXX)
> Todos los vouchers reciben un ID anónimo secuencial `VCH-XXXX` — tanto los UUID como las imágenes WhatsApp, para consistencia en el dataset.

In [5]:
vouc_counter = 1

# Archivos extra mal ubicados en PDFS/
for fname in jpg_misplaced:
    ext   = Path(fname).suffix.lower()
    nuevo = f'VCH-{vouc_counter:04d}{ext}'
    shutil.copy2(PDFS_SRC / fname, VOUC_DST / nuevo)
    mapping_rows.append({'tipo': 'VOUCHER_IMG', 'nombre_original': fname, 'nombre_anonimo': nuevo})
    print(f'Reubicado: PDFS/{fname} → VOUCHERS_anon/{nuevo}')
    vouc_counter += 1

if not jpg_misplaced:
    print('Sin archivos extra en PDFS/ — OK')

# Vouchers UUID
for fname in vouc_uuid:
    ext   = Path(fname).suffix.lower()
    nuevo = f'VCH-{vouc_counter:04d}{ext}'
    shutil.copy2(VOUC_SRC / fname, VOUC_DST / nuevo)
    mapping_rows.append({'tipo': 'VOUCHER_IMG', 'nombre_original': fname, 'nombre_anonimo': nuevo})
    vouc_counter += 1

print(f'Vouchers UUID renombrados: {len(vouc_uuid)}')

# Imágenes WhatsApp
print('\nRenombrando imágenes WhatsApp:')
for fname in vouc_whatsapp:
    ext   = Path(fname).suffix.lower()
    nuevo = f'VCH-{vouc_counter:04d}{ext}'
    shutil.copy2(VOUC_SRC / fname, VOUC_DST / nuevo)
    mapping_rows.append({'tipo': 'VOUCHER_IMG', 'nombre_original': fname, 'nombre_anonimo': nuevo})
    print(f'  {fname}  →  {nuevo}')
    vouc_counter += 1

print(f'\nTotal vouchers procesados: {vouc_counter - 1}')

Sin archivos extra en PDFS/ — OK
Vouchers UUID renombrados: 31

Renombrando imágenes WhatsApp:
  WhatsApp Image 2026-06-06 at 23.05.28.jpeg  →  VCH-0032.jpeg
  WhatsApp Image 2026-06-06 at 23.05.42.jpeg  →  VCH-0033.jpeg
  WhatsApp Image 2026-06-06 at 23.05.56.jpeg  →  VCH-0034.jpeg
  WhatsApp Image 2026-06-06 at 23.06.26.jpeg  →  VCH-0035.jpeg
  WhatsApp Image 2026-06-06 at 23.06.35.jpeg  →  VCH-0036.jpeg

Total vouchers procesados: 36


## 6 · Guardar mapping privado
> `hemogramas_mapping_PRIVADO.csv` — NO subir al repositorio. Permite recuperar la identidad original si se necesita para validación clínica.

In [6]:
with open(MAPPING, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['tipo', 'nombre_original', 'nombre_anonimo'])
    writer.writeheader()
    writer.writerows(mapping_rows)

print(f'Mapping guardado: {MAPPING}')
print(f'Total registros:  {len(mapping_rows)}')

Mapping guardado: E:\Taller Integrador I\ModelosIA\Hemogramas\hemogramas_mapping_PRIVADO.csv
Total registros:  1120


## 7 · Validación

In [7]:
pdfs_anon = sorted(os.listdir(PDFS_DST))
vouc_anon = sorted(os.listdir(VOUC_DST))

total_vouc_esperado = len(jpg_misplaced) + len(vouc_uuid) + len(vouc_whatsapp)

print('=== VALIDACIÓN ===')
print(f'PDFs en PDFS_anon/:     {len(pdfs_anon)} (esperado: {len(pdfs)})')
print(f'Imgs en VOUCHERS_anon/: {len(vouc_anon)} (esperado: {total_vouc_esperado})')
print()

sin_hg  = [f for f in pdfs_anon if not f.startswith('HG-')]
sin_vch = [f for f in vouc_anon if not f.startswith('VCH-')]
whats   = [f for f in vouc_anon if f.startswith('WhatsApp')]

print(f'PDFs sin anonimizar:          {len(sin_hg)}')
print(f'Vouchers sin renombrar a VCH: {len(sin_vch)}')
print(f'WhatsApp sin renombrar:       {len(whats)}')
print()
print('Primeros 5 PDFs anonimizados:')
for f in pdfs_anon[:5]:
    print(f'  {f}')
print()
print('Todos los vouchers en destino:')
for f in vouc_anon:
    print(f'  {f}')

=== VALIDACIÓN ===
PDFs en PDFS_anon/:     1084 (esperado: 1084)
Imgs en VOUCHERS_anon/: 36 (esperado: 36)

PDFs sin anonimizar:          0
Vouchers sin renombrar a VCH: 0
WhatsApp sin renombrar:       0

Primeros 5 PDFs anonimizados:
  HG-0001.pdf
  HG-0002.pdf
  HG-0003.pdf
  HG-0004.pdf
  HG-0005.pdf

Todos los vouchers en destino:
  VCH-0001.jpg
  VCH-0002.png
  VCH-0003.jpg
  VCH-0004.png
  VCH-0005.jpg
  VCH-0006.png
  VCH-0007.jpg
  VCH-0008.jpg
  VCH-0009.png
  VCH-0010.png
  VCH-0011.jpg
  VCH-0012.png
  VCH-0013.jpg
  VCH-0014.png
  VCH-0015.jpg
  VCH-0016.jpg
  VCH-0017.png
  VCH-0018.jpg
  VCH-0019.jpg
  VCH-0020.jpg
  VCH-0021.png
  VCH-0022.png
  VCH-0023.png
  VCH-0024.jpg
  VCH-0025.jpg
  VCH-0026.jpg
  VCH-0027.jpg
  VCH-0028.jpg
  VCH-0029.jpg
  VCH-0030.jpg
  VCH-0031.jpg
  VCH-0032.jpeg
  VCH-0033.jpeg
  VCH-0034.jpeg
  VCH-0035.jpeg
  VCH-0036.jpeg


## 8 · Verificar limpieza de metadata (muestra aleatoria)

In [8]:
import random

muestra = random.sample(pdfs_anon, min(3, len(pdfs_anon)))

print('=== VERIFICACIÓN DE METADATA (3 PDFs anonimizados) ===')
for fname in muestra:
    path = PDFS_DST / fname
    try:
        reader = PdfReader(str(path))
        meta   = reader.metadata
        campos = {k: v for k, v in (meta or {}).items() if v}
        print(f'  {fname}: metadata = {campos if campos else "(vacía) ✅"}')
    except Exception as e:
        print(f'  {fname}: error al leer ({e})')

=== VERIFICACIÓN DE METADATA (3 PDFs anonimizados) ===
  HG-0917.pdf: metadata = {'/Producer': 'pypdf'}
  HG-0697.pdf: metadata = {'/Producer': 'pypdf'}
  HG-0391.pdf: metadata = {'/Producer': 'pypdf'}


## Resumen — TA-002

| Criterio | Resultado |
|----------|-----------|
| PDFs renombrados (HG-XXXX) | 1,084 |
| Metadata PDF eliminada | ✅ |
| Vouchers UUID renombrados (VCH-XXXX) | 31 |
| Imágenes WhatsApp renombradas (VCH-XXXX) | 5 |
| Total vouchers anonimizados | 36 |
| Datos personales en destino | 0 |
| Mapping privado generado | hemogramas_mapping_PRIVADO.csv |

**Próximo paso:** TA-007 — Módulo de extracción de hemogramas (pdfplumber + EasyOCR).